In [ ]:
import pandas as pd
df = pd.read_csv('combined_data.csv')
df.head()

In [ ]:
train_data=df[['frequency_num','Time gap','pur_in_degree','new_length','eigenvector_centrality','clustering_coefficient','degree_centrality','betweenness_centrality','closeness','construct_label','dynamic_data']]
train_data.head()

In [ ]:
from sklearn.preprocessing import StandardScaler
# 对类别数据进行独热编码
df_encoded = pd.get_dummies(train_data, columns=['dynamic_data'])

# 选择连续数据特征
continuous_features = ['frequency_num','Time gap','pur_in_degree','new_length','eigenvector_centrality','clustering_coefficient','degree_centrality','betweenness_centrality','closeness','construct_label']

# 标准化连续数据
scaler = StandardScaler()
df_encoded[continuous_features] = scaler.fit_transform(df_encoded[continuous_features])

df_encoded.head()

In [ ]:
import pandas as pd
import numpy as np

# 计算每一列的熵
def entropy(series):
    p = series / series.sum()
    return -np.sum(p * np.log(p))

In [ ]:
def entropy_data(df):
    entropies = df.apply(entropy)

    # 计算每一列的权重
    weights = 1 - entropies / entropies.sum()

    # 将权重应用于数据并合并为一列
    weighted_sum = (df * weights).sum(axis=1)

    # 添加合并后的列到DataFrame
    df['Merged'] = weighted_sum

    return(df)

In [ ]:
M1=df_encoded[['pur_in_degree','new_length']]
M2=df_encoded[['eigenvector_centrality','clustering_coefficient','degree_centrality']]
M3=df_encoded[['betweenness_centrality','closeness','construct_label']]

In [ ]:
M1=entropy_data(M1)
M2=entropy_data(M2)
M3=entropy_data(M3)

In [ ]:
after_M1=M1[['Merged']]
after_M2=M2[['Merged']]
after_M3=M3[['Merged']]

In [ ]:
last_data=pd.concat([df_encoded[['frequency_num','Time gap']],after_M1,after_M2,after_M3,df_encoded[['dynamic_data_增长','dynamic_data_稳定','dynamic_data_衰退']]], axis=1)
last_data.columns=['F','R','M1','M2','M3','T1','T2','T3']
last_data.head()

In [ ]:
temp_df=last_data[['F','R','M1','M2','M3']]

In [ ]:
# 计算每列的平均值
mean_values = temp_df.mean()

# 比较每列的值和平均值，并将大于平均值的元素设为1，小于平均值的元素设为0
result = temp_df.apply(lambda x: x > mean_values[x.name], axis=0).astype(int)

result

In [ ]:
df_encoded.iloc[:, -3:]

In [ ]:
result=pd.concat([result,df_encoded.iloc[:, -3:]],axis=1)
result

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import calinski_harabasz_score,davies_bouldin_score

sse=[]
CH_data=[]
DB_data=[]

for i in range(2,20):
    kmeans = KMeans(n_clusters=i,init='k-means++',max_iter=300,n_init=10,random_state=0)
    result_list =kmeans.fit_predict(result)
    
    CH_index=calinski_harabasz_score(result,result_list)
    DB_index=davies_bouldin_score(result,result_list)
    
    print(f"方差比为: {CH_index}")
    print(f"DB值为: {DB_index}")
    print('SSE值为:',kmeans.inertia_)
    
    sse.append(kmeans.inertia_)
    CH_data.append(CH_index)
    DB_data.append(DB_index)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(range(2,20),sse)
plt.title('The Elbow Method')
plt.xlabel('Number of clusters')
plt.ylabel('WCSS')
plt.show()

In [ ]:
plt.plot(range(2,20),CH_data)
plt.title('The Elbow Method')
plt.xlabel('Number of clusters')
plt.ylabel('CH_data')
plt.show()

In [ ]:
import torch

train_data=torch.from_numpy(np.array(result,dtype=float)).float()

In [ ]:
from kmodes.kmodes import KModes
import matplotlib.pyplot as plt
from sklearn import metrics  # Import the metrics module

# 定义聚类个数范围
cluster_range = range(2, 10)
sse1 = []
silhouette_scores1 = []

# 计算不同聚类个数下的SSE
for k in cluster_range:
    km = KModes(n_clusters=k, init='Huang', n_init=5, verbose=0)
    labels =km.fit_predict(train_data)
    
    print(km.cost_)
    print(metrics.silhouette_score(train_data, labels, metric='hamming'))
    
    silhouette_scores1.append(metrics.silhouette_score(train_data, labels, metric='hamming'))
    sse1.append(km.cost_)
    

In [ ]:
# 绘制肘部法则图形
plt.plot(cluster_range, sse, marker='o')
plt.xlabel('Number of clusters')
plt.ylabel('SSE')
plt.title('Elbow Method for Optimal k')
plt.show()